# Retrieval-augmented generation over geotechnical reports

Exercise: [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32-courses/ai-geotech/blob/main/docs/07-llm/07b-llm-rag-geotechnical-exercise.ipynb) Solution: [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32-courses/ai-geotech/blob/main/docs/07-llm/07b-llm-rag-geotechnical.ipynb)

The model in 7a answered from memory. Here it answers from four site investigation reports.
We read the pages, cut them into chunks, embed the chunks as vectors, and store them in a
local database. A question retrieves the nearest chunks, and those chunks go into the prompt
as the only material the model may use.

Run this cell first. It installs the packages, reads your API key, and names the model.
Locally the key comes from a `.env` file in this folder. On Colab it comes from
Settings > Secrets, or from a prompt.

In [ ]:
!pip install -q openai pypdf chromadb tiktoken python-dotenv
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()                                   # local: reads .env in this folder if present
if not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata       # Colab: Settings > Secrets > OPENAI_API_KEY
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
client = OpenAI()
MODEL = "gpt-5.6-luna"   # $0.20 in / $1.20 out per 1M tokens, developers.openai.com/api/docs/pricing, 2026-09-10

The four site investigation reports live in `docs/` in the repository. On Colab there is no
repository, so this cell downloads `docs.zip` from GitHub and unpacks it into `reports/`.

In [ ]:
REPORTS = Path("docs") if Path("docs").exists() else Path("reports")   # repo checkout vs Colab
if not REPORTS.exists():
    !wget -nc https://raw.githubusercontent.com/kks32-courses/ai-geotech/main/docs/07-llm/docs.zip
    !unzip -n -q docs.zip -d reports
pdf_paths = sorted(REPORTS.glob("*.pdf"))
print([p.name for p in pdf_paths])

`pypdf` reads text page by page. Keeping the document name and page number with each page is
what lets the answer cite a source. Pages with almost no text are scanned images, and nothing
downstream can read them.

In [ ]:
from pypdf import PdfReader

pages = []
for path in pdf_paths:
    for n, page in enumerate(PdfReader(str(path)).pages, start=1):
        pages.append({"doc": path.name, "page": n, "text": page.extract_text() or ""})

print(f"{len(pages)} pages from {len(pdf_paths)} documents")
short = [p for p in pages if len(p["text"]) < 50]
print(f"{len(short)} pages with fewer than 50 characters, scanned or empty:")
for p in short[:20]:
    print(f'  {p["doc"]} p.{p["page"]}')

A whole page is too coarse to retrieve well and a sentence is too fine. We cut each page into
windows of 400 tokens with 50 tokens of overlap, so a fact split across a window boundary still
appears whole in one of the two windows. `cl100k_base` is the tokenizer the embedding model uses.

In [ ]:
import tiktoken

ENCODER = tiktoken.get_encoding("cl100k_base")


def chunk(pages, size=400, overlap=50):
    """Cut each page into windows of `size` tokens that overlap by `overlap` tokens."""
    out = []
    for p in pages:
        tokens = ENCODER.encode(p["text"])
        start = 0
        while start < len(tokens):
            end = min(len(tokens), start + size)
            out.append({"doc": p["doc"], "page": p["page"], "text": ENCODER.decode(tokens[start:end])})
            if end == len(tokens):
                break
            start = end - overlap
    return out


chunks = chunk(pages)
print(f"{len(chunks)} chunks")

The embedding model turns each chunk into a vector. Chroma stores the vectors on disk in
`chroma_db/` and compares them by cosine distance. The build runs only when the collection is
empty, so rerunning this cell costs nothing. The four reports come to about 132,000 chunk
tokens, which is a third of a cent at $0.02 per 1M tokens.

In [ ]:
import chromadb

chroma = chromadb.PersistentClient(path="chroma_db")
collection = chroma.get_or_create_collection("reports", metadata={"hnsw:space": "cosine"})

if collection.count() == 0:
    for start in range(0, len(chunks), 64):
        batch = chunks[start:start + 64]
        vectors = client.embeddings.create(
            model="text-embedding-3-small",
            input=[c["text"] for c in batch],
        )
        collection.add(
            ids=[f'{c["doc"]}-p{c["page"]}-c{start + i}' for i, c in enumerate(batch)],
            embeddings=[d.embedding for d in vectors.data],
            documents=[c["text"] for c in batch],
            metadatas=[{"doc": c["doc"], "page": c["page"]} for c in batch],
        )
        print(f"embedded {start + len(batch)} of {len(chunks)} chunks")

print(f"{collection.count()} chunks in the collection")

Retrieval embeds the question with the same model and asks the database for the nearest chunks.
Chroma reports cosine distance, so similarity is one minus that. A similarity near 0.5 is a
strong match for this corpus.

In [ ]:
def search_reports(query, k=5):
    """Return the k chunks closest to `query`, each with its document, page, and similarity."""
    vector = client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding
    results = collection.query(
        query_embeddings=[vector],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    if not results["documents"] or not results["documents"][0]:
        return []
    return [
        {"text": text, "doc": meta["doc"], "page": meta["page"], "similarity": 1 - distance}
        for text, meta, distance in zip(
            results["documents"][0], results["metadatas"][0], results["distances"][0]
        )
    ]


for hit in search_reports("allowable bearing pressure for spread footings"):
    print(f'{hit["doc"]:<38} p.{hit["page"]:<4} similarity {hit["similarity"]:.3f}')

The retrieved chunks go into the prompt with a label on each. The instructions confine the model
to those chunks and require a citation, so every claim in the answer points back to a page you
can open.

In [ ]:
def ask(question, k=5):
    hits = search_reports(question, k=k)
    if not hits:
        print("No sources found.")
        return
    sources = "\n\n".join(
        f'[Source {i + 1}] {h["doc"]} p.{h["page"]}: {h["text"]}' for i, h in enumerate(hits)
    )
    resp = client.responses.create(
        model=MODEL,
        instructions="Answer from the sources only. Cite sources as [Source i]. If the sources do not contain the answer, say so.",
        input=f"Sources:\n{sources}\n\nQuestion: {question}",
    )
    print(resp.output_text)
    print()
    for i, h in enumerate(hits):
        print(f'[Source {i + 1}] {h["doc"]} p.{h["page"]}')

Section 4.3.1 of `TERRACON_FINALV5.pdf` recommends 2,000 psf for footing widths under 9 feet,
with a minimum embedment of 18 inches below finished grade. Check the answer against the pages
it cites.

In [ ]:
ask("What allowable bearing pressure does the Terracon report recommend for shallow spread footings, and what minimum embedment depth?")